# Deep Learning 040 — Why CNNs Exist

The lesson's claim is arithmetic, not taste: a dense first layer on a 224×224
colour image needs **150,529,000** parameters where a convolutional layer needs
**896** — about **168,000×** fewer. This notebook recomputes all of it.

| Part | What we check |
|---|---|
| A | the dense count, and where it comes from |
| B | the conv count, and why the image size does not appear in it |
| C | the same comparison at 1080p |
| D | the second objection: flattening destroys adjacency |

In [ ]:
import numpy as np

## Part A — A dense first layer

In [ ]:
h, w, c = 224, 224, 3
units = 1000
inputs = h * w * c
dense_params = inputs * units + units
print(f"{h}x{w}x{c} = {inputs:,} input values")
print(f"dense layer of {units:,} units: {inputs:,} x {units:,} + {units:,} = {dense_params:,}")
assert dense_params == 150_529_000

## Part B — A convolutional layer

In [ ]:
filters, k = 32, 3
conv_params = (k * k * c) * filters + filters
print(f"{filters} filters of {k}x{k}x{c}: {k*k*c} weights each + 1 bias = {conv_params}")
assert conv_params == 896
print(f"\nratio: {dense_params / conv_params:,.0f}x fewer parameters")
assert round(dense_params / conv_params / 1000) == 168

## Part C — Now make the image bigger

The dense count scales with the number of pixels. The conv count does not appear
to notice.

In [ ]:
print(f"{'image':>12}{'dense params':>18}{'conv params':>14}")
for (H, W) in ((224, 224), (512, 512), (1920, 1080)):
    d = H * W * c * units + units
    v = (k * k * c) * filters + filters
    print(f"{f'{H}x{W}':>12}{d:>18,}{v:>14,}")
assert (k * k * c) * filters + filters == 896     # unchanged at every size
print("\nThe filter is the parameter. It is reused at every position, so the")
print("image size changes the amount of computation, not the number of weights.")

## Part D — The other objection

Parameter count is only half the argument. Flattening throws away which pixels
were neighbours — the point lesson 012 made with MNIST.

In [ ]:
img = np.arange(16).reshape(4, 4)
print("image:\n", img)
print("\nneighbours of pixel (1,1) in the image:",
      [int(img[1, 0]), int(img[0, 1]), int(img[1, 2]), int(img[2, 1])])

flat = img.flatten()
i = 1 * 4 + 1
print("after flatten, its neighbours in the vector:", [int(flat[i - 1]), int(flat[i + 1])])
print("\nThe vertical neighbours are now 4 positions away, and nothing in a dense")
print("layer knows that 4 is special. A convolution keeps the grid.")

## What to take away

- **224×224×3 into 1,000 dense units: 150,529,000 parameters.** One layer.
- **32 conv filters of 3×3×3: 896 parameters**, about **168,000×** fewer.
- The conv count is **independent of image size** — the weights live in the filter.
- Flattening also destroys adjacency, which is the second reason dense layers
  are wrong for images.

## Exercises

1. How many parameters would the dense layer need at 1920×1080? Compare with the
   conv layer's 896.
2. Raise the filter size from 3×3 to 7×7. How much does the conv count grow, and
   is it still negligible?
3. A conv layer with 64 filters on a 1-channel (grayscale) image: how many
   parameters?